In [13]:
import torch
import math
import os
import random
torch.manual_seed(35)
random.seed(35)
import json

In [2]:
def cot_trace(a, b):
    da = [int(c) for c in str(a)][::-1]
    db = [int(c) for c in str(b)][::-1]
    n = max(len(da), len(db))
    da += [0] * (n - len(da))
    db += [0] * (n - len(db))
    steps, carry = [], 0
    for i in range(n):
        s = da[i] + db[i] + carry
        new_carry = s // 10
        if carry:
            steps.append(f'{da[i]}+{db[i]}+{carry}={s}c{new_carry}')
        else:
            steps.append(f'{da[i]}+{db[i]}={s}c{new_carry}')
        carry = new_carry
    return ','.join(steps)

MAX_DIGITS = 2
lines = []
for _ in range(2000):
    a = random.randint(0, 10**MAX_DIGITS - 1)
    b = random.randint(0, 10**MAX_DIGITS - 1)
    lines.append(f'{a}+{b}=|{cot_trace(a, b)}|{a+b}')

corpus = '\n'.join(lines)
print(corpus[:300])  # preview

# Character-level vocab (each character is a token). Digits and symbols each become their own id.
chars = sorted(set(corpus))
vocab = chars
char_to_index = {c: i for i, c in enumerate(vocab)}
index_to_char = {i: c for c, i in char_to_index.items()}
vocab_size = len(vocab)
vocab_size

70+42=|0+2=2c0,7+4=11c1|112
96+16=|6+6=12c1,9+1+1=11c1|112
95+43=|5+3=8c0,9+4=13c1|138
19+36=|9+6=15c1,1+3+1=5c0|55
55+32=|5+2=7c0,5+3=8c0|87
72+7=|2+7=9c0,7+0=7c0|79
95+67=|5+7=12c1,9+6+1=16c1|162
96+35=|6+5=11c1,9+3+1=13c1|131
83+46=|3+6=9c0,8+4=12c1|129
91+74=|1+4=5c0,9+7=16c1|165
96+12=|6+2=8c0,


16

In [3]:
corpus

'70+42=|0+2=2c0,7+4=11c1|112\n96+16=|6+6=12c1,9+1+1=11c1|112\n95+43=|5+3=8c0,9+4=13c1|138\n19+36=|9+6=15c1,1+3+1=5c0|55\n55+32=|5+2=7c0,5+3=8c0|87\n72+7=|2+7=9c0,7+0=7c0|79\n95+67=|5+7=12c1,9+6+1=16c1|162\n96+35=|6+5=11c1,9+3+1=13c1|131\n83+46=|3+6=9c0,8+4=12c1|129\n91+74=|1+4=5c0,9+7=16c1|165\n96+12=|6+2=8c0,9+1=10c1|108\n46+1=|6+1=7c0,4+0=4c0|47\n65+1=|5+1=6c0,6+0=6c0|66\n12+92=|2+2=4c0,1+9=10c1|104\n63+47=|3+7=10c1,6+4+1=11c1|110\n93+3=|3+3=6c0,9+0=9c0|96\n43+86=|3+6=9c0,4+8=12c1|129\n63+6=|3+6=9c0,6+0=6c0|69\n2+30=|2+0=2c0,0+3=3c0|32\n41+10=|1+0=1c0,4+1=5c0|51\n20+43=|0+3=3c0,2+4=6c0|63\n21+72=|1+2=3c0,2+7=9c0|93\n9+61=|9+1=10c1,0+6+1=7c0|70\n46+10=|6+0=6c0,4+1=5c0|56\n37+20=|7+0=7c0,3+2=5c0|57\n73+9=|3+9=12c1,7+0+1=8c0|82\n86+16=|6+6=12c1,8+1+1=10c1|102\n69+93=|9+3=12c1,6+9+1=16c1|162\n56+63=|6+3=9c0,5+6=11c1|119\n41+50=|1+0=1c0,4+5=9c0|91\n79+49=|9+9=18c1,7+4+1=12c1|128\n4+33=|4+3=7c0,0+3=3c0|37\n99+76=|9+6=15c1,9+7+1=17c1|175\n98+58=|8+8=16c1,9+5+1=15c1|156\n64+71=|4+1=5c0,6+7=1

In [4]:
embed_dim = 128
number_of_heads = 8
head_size = embed_dim // number_of_heads
ffn_hidden = 512

In [5]:
def pos_enc(seq_len):
    pe = torch.zeros(seq_len, embed_dim)
    for pos in range(seq_len):
        for i in range(0, embed_dim, 2):
            pe[pos, i]   = math.sin(pos / 10000 ** (i / embed_dim))
            pe[pos, i+1] = math.cos(pos / 10000 ** (i / embed_dim))
    return pe

In [6]:
class DecoderBlock(torch.nn.Module):
    def __init__(self, embed_dim, num_heads, ffn_hidden):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = embed_dim // num_heads
        w = lambda *shape: torch.nn.Parameter(torch.randn(*shape) * 0.02)
        self.wq = w(embed_dim, embed_dim)
        self.wk = w(embed_dim, embed_dim)
        self.wv = w(embed_dim, embed_dim)
        self.wo = w(embed_dim, embed_dim)
        self.ff1_w = w(embed_dim, ffn_hidden)
        self.ff1_b = torch.nn.Parameter(torch.zeros(ffn_hidden))
        self.ff2_w = w(ffn_hidden, embed_dim)
        self.ff2_b = torch.nn.Parameter(torch.zeros(embed_dim))
        self.norm1 = torch.nn.LayerNorm(embed_dim)
        self.norm2 = torch.nn.LayerNorm(embed_dim)

    def split(self, m):
        return m.view(m.shape[0], self.num_heads, self.head_size).transpose(0, 1)

    def forward(self, x):
        T, D = x.shape
        mask = torch.triu(torch.full((T, T), float('-inf')), diagonal=1)
        #split = lambda m: m.view(T, self.num_heads, self.head_size).transpose(0, 1)
        Q, K, V = self.split(x @ self.wq), self.split(x @ self.wk), self.split(x @ self.wv)
        attention = torch.softmax(Q @ K.transpose(-2, -1) / math.sqrt(self.head_size) + mask, dim=-1)
        out = (attention @ V).transpose(0, 1).contiguous().view(T, D) @ self.wo
        x = self.norm1(x + out)
        ffn = torch.relu(x @ self.ff1_w + self.ff1_b) @ self.ff2_w + self.ff2_b
        return self.norm2(x + ffn)


class ReasoningModel(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=8, ffn_hidden=512, num_layers=1, max_len=64):
        super().__init__()
        self.embedding = torch.nn.Parameter(torch.randn(vocab_size, embed_dim))
        self.blocks = torch.nn.ModuleList(
            DecoderBlock(embed_dim, num_heads, ffn_hidden) for _ in range(num_layers))
        self.out_w = torch.nn.Parameter(torch.randn(embed_dim, vocab_size) * 0.02)
        self.out_b = torch.nn.Parameter(torch.zeros(vocab_size))
        self.register_buffer('pos', pos_enc(max_len))   # computed once, not every call

    def forward(self, tokens):
        x = self.embedding[tokens] + self.pos[:len(tokens)]
        for block in self.blocks:
            x = block(x)
        return x @ self.out_w + self.out_b

In [7]:
training_sequences = []
for line in corpus.strip().split('\n'):
    line = line + '\n'
    ids = [char_to_index[c] for c in line]
    source_token = torch.tensor(ids[:-1])
    target_token = torch.tensor(ids[1:])
    training_sequences.append((source_token, target_token))

print(f'Training sequences: {len(training_sequences)}')
print('First sequence chars:', ''.join(index_to_char[i.item()] for i in training_sequences[0][0]))

Training sequences: 2000
First sequence chars: 70+42=|0+2=2c0,7+4=11c1|112


In [8]:
for source_tokens, target_tokens in training_sequences:
    print(''.join(index_to_char[t.item()] for t in source_tokens))

70+42=|0+2=2c0,7+4=11c1|112
96+16=|6+6=12c1,9+1+1=11c1|112
95+43=|5+3=8c0,9+4=13c1|138
19+36=|9+6=15c1,1+3+1=5c0|55
55+32=|5+2=7c0,5+3=8c0|87
72+7=|2+7=9c0,7+0=7c0|79
95+67=|5+7=12c1,9+6+1=16c1|162
96+35=|6+5=11c1,9+3+1=13c1|131
83+46=|3+6=9c0,8+4=12c1|129
91+74=|1+4=5c0,9+7=16c1|165
96+12=|6+2=8c0,9+1=10c1|108
46+1=|6+1=7c0,4+0=4c0|47
65+1=|5+1=6c0,6+0=6c0|66
12+92=|2+2=4c0,1+9=10c1|104
63+47=|3+7=10c1,6+4+1=11c1|110
93+3=|3+3=6c0,9+0=9c0|96
43+86=|3+6=9c0,4+8=12c1|129
63+6=|3+6=9c0,6+0=6c0|69
2+30=|2+0=2c0,0+3=3c0|32
41+10=|1+0=1c0,4+1=5c0|51
20+43=|0+3=3c0,2+4=6c0|63
21+72=|1+2=3c0,2+7=9c0|93
9+61=|9+1=10c1,0+6+1=7c0|70
46+10=|6+0=6c0,4+1=5c0|56
37+20=|7+0=7c0,3+2=5c0|57
73+9=|3+9=12c1,7+0+1=8c0|82
86+16=|6+6=12c1,8+1+1=10c1|102
69+93=|9+3=12c1,6+9+1=16c1|162
56+63=|6+3=9c0,5+6=11c1|119
41+50=|1+0=1c0,4+5=9c0|91
79+49=|9+9=18c1,7+4+1=12c1|128
4+33=|4+3=7c0,0+3=3c0|37
99+76=|9+6=15c1,9+7+1=17c1|175
98+58=|8+8=16c1,9+5+1=15c1|156
64+71=|4+1=5c0,6+7=13c1|135
96+5=|6+5=11c1,9+0+1=10c1|1

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using', device)

model = ReasoningModel(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss()

logits = model(source_tokens)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

Using cuda


tensor(0.)

In [10]:
epochs = 20
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    random.shuffle(training_sequences)
    for source_tokens, target_tokens in training_sequences:
        logits = model(source_tokens)
        loss = loss_fn(logits, target_tokens)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
    print('epoch', epoch, 'avg_loss', total_loss / len(training_sequences))

epoch 1 avg_loss 0.8892613456398248
epoch 2 avg_loss 0.44079192051291466
epoch 3 avg_loss 0.3613328344896436
epoch 4 avg_loss 0.3360934496894479
epoch 5 avg_loss 0.31628561484068635
epoch 6 avg_loss 0.3061933047622442
epoch 7 avg_loss 0.3064318375661969
epoch 8 avg_loss 0.2976586074978113
epoch 9 avg_loss 0.29606201757490636
epoch 10 avg_loss 0.29312995591759683
epoch 11 avg_loss 0.29300992003828286
epoch 12 avg_loss 0.2842951240092516
epoch 13 avg_loss 0.2863628509566188
epoch 14 avg_loss 0.28135475757718087
epoch 15 avg_loss 0.2867764079943299
epoch 16 avg_loss 0.2804185966253281
epoch 17 avg_loss 0.27980052304267883
epoch 18 avg_loss 0.27878688314557076
epoch 19 avg_loss 0.27826515451073647
epoch 20 avg_loss 0.27712922094762327


In [11]:
torch.save({
    'state_dict': model.state_dict(),          # all the trained weights
    'config': {                                # how to rebuild the model
        'vocab_size': vocab_size,
        'embed_dim': embed_dim,
        'num_heads': number_of_heads,
        'ffn_hidden': ffn_hidden,
        'num_layers': len(model.blocks),
        'max_len': model.pos.shape[0],
    },
    'char_to_index': char_to_index,            # the vocabulary
    'index_to_char': index_to_char,
}, 'reasoning_model.pth')
print('Saved.')

Saved.


In [14]:
checkpoint = torch.load('reasoning_model.pth', map_location='cpu')
model = ReasoningModel(**checkpoint['config'])
model.load_state_dict(checkpoint['state_dict'])
model.eval()

scripted = torch.jit.script(model)
torch.jit.save(scripted, 'reasoning_model_full.pt',
               _extra_files={'vocab.json': json.dumps(checkpoint['char_to_index'])})
print('Saved reasoning_model_full.pt')

Saved reasoning_model_full.pt


In [15]:
def generate(prompt, max_chars=60, temperature=0.0):
    unknown = [c for c in prompt if c not in char_to_index]
    if unknown:
        print(f'Unknown chars: {unknown}')
        return

    generated = list(prompt)

    model.eval()
    with torch.no_grad():
        for _ in range(max_chars):
            tokens = torch.tensor([char_to_index[c] for c in generated], device=model.embedding.device)
            logits = model(tokens)
            if temperature == 0.0:
                next_token = logits[-1].argmax().item()
            else:
                probs = torch.softmax(logits[-1] / temperature, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
            next_char = index_to_char[next_token]
            if next_char == '\n':
                break
            generated.append(next_char)

    output = ''.join(generated)
    print(f'{prompt}   ->   {output}')
    return output

# test
generate('27+45=')
generate('9+8=')
generate('50+50=')

27+45=   ->   27+45=|7+5=12c1,2+4+1=7c0|72
9+8=   ->   9+8=|9+8=17c1|16
50+50=   ->   50+50=|0+0=0c0,5+5=10c1|100


'50+50=|0+0=0c0,5+5=10c1|100'